# Hub Urgency — End-to-End Run

Set the **input folder** and the **output folder** in the next cell, then *Run All*.

The notebook runs both stages:

1. **Link** — which inventar projects intersect each hub's H3 cells (`InventarHubLinker`)
2. **Score** — attach project status, apply manual overrides, compute each hub's weighted progress (`HubProjectStatusPipeline`)

### What the input folder should contain

| File | Required | Notes |
|------|----------|-------|
| Hubs CSV | yes | Auto-detected: the CSV with `group` and `h3_index` columns |
| Inventar | yes | One `*.geojson` file **or** `geom_point.shp` / `geom_line.shp` / `geom_multiline.shp` (in the folder or one sub-folder) |
| `status_weights.csv` | yes | Columns `Proj_status`, `weight`. Labels must match the inventar's `Proj_status` exactly (e.g. the Hebrew labels in a GeoJSON). If missing, a pre-filled `status_weights_template.csv` is written to the output folder |
| `status_overrides.csv` | no | Columns `uid`, `Proj_status`. **If the file exists**, the statuses of those UIDs are replaced; **if not**, the statuses from the GeoJSON/shapefile are used as they are |
| `data.csv` | no | Project attributes (`uid`, `proj_name`, `main_type`, `Proj_status`, `scn_year`). Needed only if the inventar itself does not carry them (typical for shapefiles) |

## 1. Setup

In [ ]:
try:
    import huburgency
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q git+https://github.com/oh-da/HubUrgency.git
    import huburgency

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from huburgency import discover_inputs, run_end_to_end

logging.getLogger().setLevel(logging.INFO)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Google Colab only: uncomment to read/write Google Drive folders.
# from google.colab import drive; drive.mount('/content/drive')

## 2. Configuration

In [ ]:
INPUT_DIR = Path(r'/content/drive/MyDrive/Hubs_Urgency/input')    # <- folder with the input files
OUTPUT_DIR = Path(r'/content/drive/MyDrive/Hubs_Urgency/output')  # <- results are written here

# Optional - leave as is unless your file names differ (names are relative to INPUT_DIR).
HUBS_CSV = None                            # None = auto-detect
INVENTAR = None                            # None = auto-detect (.geojson file or geom_*.shp)
STATUS_WEIGHTS_CSV = 'status_weights.csv'
STATUS_OVERRIDES_CSV = 'status_overrides.csv'   # optional file
PROJECT_DATA_CSV = 'data.csv'                   # optional file
ENCODING = 'windows-1255'                  # hubs CSV + written CSVs (falls back to UTF-8 if a character does not fit)
GROUP_COL = 'group'

## 3. Check the inputs

In [ ]:
inputs = discover_inputs(
    INPUT_DIR, hubs_csv=HUBS_CSV, inventar=INVENTAR,
    status_weights_csv=STATUS_WEIGHTS_CSV, status_overrides_csv=STATUS_OVERRIDES_CSV,
    project_data_csv=PROJECT_DATA_CSV,
)
print(f"Hubs CSV        : {inputs.hubs_csv}")
print(f"Inventar        : {inputs.inventar}")
print(f"Status weights  : {inputs.status_weights_csv}  (exists: {inputs.status_weights_csv.exists()})")
print(f"Status overrides: {inputs.status_overrides_csv or 'none found -> original statuses are used'}")
print(f"Project data    : {inputs.project_data_csv or 'none found -> read from the inventar'}")

## 4. Run
Stage 1 is the slow step (one spatial query per hub cell). If `status_weights.csv` is missing, this cell stops **before** Stage 1 and writes `status_weights_template.csv` to the output folder: review the weights, save the file into the input folder and run again.

In [ ]:
result = run_end_to_end(
    INPUT_DIR, OUTPUT_DIR,
    hubs_csv=HUBS_CSV, inventar=INVENTAR,
    status_weights_csv=STATUS_WEIGHTS_CSV, status_overrides_csv=STATUS_OVERRIDES_CSV,
    project_data_csv=PROJECT_DATA_CSV,
    encoding=ENCODING, group_col=GROUP_COL,
)

## 5. Status checks

In [ ]:
projects = result.project_df
print(f"Projects in the inventar : {len(projects):,}")
print(f"Statuses overridden      : {result.n_overridden:,}")
if result.n_overridden:
    changed = projects.loc[projects['status_overridden'],
                           ['uid', 'proj_name', 'Proj_status_original', 'Proj_status']]
    display(changed)

if result.unmapped_statuses:
    print(f"\n⚠ Statuses with NO weight (counted as 0): {result.unmapped_statuses}")
    print("  Add them to status_weights.csv (see status_weights_template.csv in the output folder).")
else:
    print("\n✓ Every status has a weight")

display(projects['Proj_status'].value_counts().rename('projects').to_frame())

## 6. Results

In [ ]:
progress = result.progress_df
print(f"Hubs: {len(progress):,}   (with projects: {(progress['total_projects'] > 0).sum():,})")
print(progress['status_progress_pct'].describe().round(2).to_string())
display(progress.sort_values('status_progress_pct', ascending=False).head(15))

In [ ]:
with_projects = progress[progress['total_projects'] > 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(with_projects['status_progress_pct'], bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(with_projects['status_progress_pct'].mean(), color='red', linestyle='--', label='Mean')
axes[0].set(xlabel='Status progress (%)', ylabel='Hubs', title='Hub status progress (hubs with projects)')
axes[0].legend()
axes[1].scatter(with_projects['total_projects'], with_projects['status_progress_pct'], alpha=0.6)
axes[1].set(xlabel='Projects linked to hub', ylabel='Status progress (%)', title='Progress vs. project count')
plt.tight_layout()
plt.show()

## 7. Output files

In [ ]:
for name, path in result.outputs.items():
    print(f"{name:18s} {path}")